# 🎬 YT Short Clipper Pro — Streamlit + Ngrok

**Arsitektur mirip MoneyPrinterTurbo**: Streamlit WebUI + Ngrok Tunneling + GPU T4

---

### Fitur:
- 🌐 **WebUI via Ngrok** — Akses dari browser HP/laptop
- 🤖 **AI Analysis** — Gemini/Groq/OpenRouter
- 👁️ **Face Tracking** + Karaoke Subtitle
- 🎨 **B-Roll Overlay** dari Pexels
- 🎵 **Background Music** auto-download
- 🖥️ **GPU T4** untuk Whisper + FFmpeg

### ⚠️ Yang Tidak Tersedia:
- **Voice Hook** — butuh Voicebox running di komputer lokal

---

**Cara pakai:** Jalankan cell satu per satu dari atas ke bawah.

## 📌 Cell 1: Mount Google Drive & Setup Dirs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/YTShortClipper/output', exist_ok=True)
os.makedirs('/content/temp', exist_ok=True)
print('✅ Google Drive mounted!')

## 📌 Cell 2: Install Dependencies + Pyngrok

In [ ]:
# System deps
!apt-get -qq install ffmpeg

# Python deps
!pip install -q yt-dlp[default] opencv-python-headless numpy Pillow requests mediapipe python-dotenv
!pip install -q faster-whisper google-genai
!pip install -q streamlit pyngrok

print('✅ Semua dependencies terinstall!')

## 📌 Cell 3: Clone Repository

In [ ]:
!git clone https://github.com/Chukie99/yt-short-clipper-offline.git /content/yt-short-clipper-offline 2>/dev/null || true

import sys, os
repo_dir = '/content/yt-short-clipper-offline'
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
os.chdir(repo_dir)

# Verify
for f in ['clipper_core.py', 'app.py', 'requirements.txt']:
    status = '✅' if os.path.exists(os.path.join(repo_dir, f)) else '❌'
    print(f'  {status} {f}')

# Setup directories for Colab
from clipper_core import setup_directories
setup_directories(
    temp_dir='/content/temp',
    output_dir='/content/drive/MyDrive/YTShortClipper/output',
    config_file='/content/drive/MyDrive/YTShortClipper/config.json',
)
print('\n✅ Repository ready!')

## 📌 Cell 4: Setup API Keys + Ngrok Token

### Cara Setup (Pilih salah satu):

**Metode 1 — Colab Secrets (Recommended):**
1. Klik ikon 🔑 di sidebar kiri notebook
2. Klik "Add new secret"
3. Tambah: `GEMINI_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`, `NGROK_AUTH_TOKEN`
4. Aktifkan "Notebook access" untuk tiap key

**Metode 2 — Manual input:**
- Cell ini akan meminta input lewat `getpass`

In [ ]:
import os

# Try Colab Secrets first
try:
    from google.colab import userdata
    
    for key in ['GEMINI_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY', 'NGROK_AUTH_TOKEN']:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f'  ✅ {key} loaded from Secrets')
    
    if not any(os.environ.get(k) for k in ['GEMINI_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY']):
        print('⚠️ No API keys in Secrets. Using getpass...')
        raise Exception('No keys')
        
except Exception:
    import getpass
    print('Masukkan API keys (skip jika tidak punya):')
    
    for key in ['GEMINI_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY', 'NGROK_AUTH_TOKEN']:
        if not os.environ.get(key):
            label = key.replace('_', ' ').title()
            k = getpass.getpass(f'{label}: ')
            if k: os.environ[key] = k

# Verify
has_api = any(os.environ.get(k) for k in ['GEMINI_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY'])
has_ngrok = bool(os.environ.get('NGROK_AUTH_TOKEN'))
print(f'\n{"✅" if has_api else "⚠️"} API Keys: {"Ready" if has_api else "Missing"}')
print(f'{"✅" if has_ngrok else "⚠️"} Ngrok: {"Ready" if has_ngrok else "Missing — will use Colab direct link"}')

## 📌 Cell 5: Launch Streamlit + Ngrok 🚀

Cell ini akan:
1. Jalankan Streamlit `app.py` di background
2. Buat Ngrok tunnel (atau Colab direct link)
3. Tampilkan link publik yang bisa langsung diklik

In [ ]:
import subprocess, time, os, sys

repo_dir = '/content/yt-short-clipper-offline'
os.chdir(repo_dir)

# Kill existing streamlit
!pkill -f streamlit 2>/dev/null || true
time.sleep(2)

# Start Streamlit in background
process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.address=0.0.0.0',
     '--browser.gatherUsageStats=false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for Streamlit to start
print('⏳ Starting Streamlit...')
for i in range(15):
    time.sleep(1)
    if process.poll() is not None:
        print(f'❌ Streamlit crashed! Exit code: {process.returncode}')
        break
    print(f'  Waiting... ({i+1}s)')

# Setup Ngrok or Colab link
ngrok_token = os.environ.get('NGROK_AUTH_TOKEN', '')
if ngrok_token:
    try:
        from pyngrok import ngrok, conf
        conf.get_default().auth_token = ngrok_token
        ngrok.kill()
        tunnel = ngrok.connect(8501, 'http')
        public_url = tunnel.public_url
        print(f'\n{"="*60}')
        print(f'🌐 PUBLIC URL (Ngrok): {public_url}')
        print(f'{"="*60}')
        print(f'Buka link di atas untuk akses WebUI dari browser mana saja!')
    except Exception as e:
        print(f'⚠️ Ngrok error: {e}')
        print(f'\n🌐 Colab direct: https://localhost:8501')
        print(f'(Atau klik tab "Sites" di samping notebook)')
else:
    print(f'\n{"="*60}')
    print(f'🌐 Colab Direct Link: http://localhost:8501')
    print(f'{"="*60}')
    print(f'Klik tab "Sites" di samping notebook untuk akses.')
    print(f'Atau buka: https://localhost:8501')

## 📌 Cell 6: Monitor Logs (Opsional)

Jalankan cell ini untuk melihat log Streamlit secara real-time.

In [ ]:
# Check if streamlit is still running
!ps aux | grep streamlit | grep -v grep

# Check output directory
import os
from pathlib import Path
output_dir = Path('/content/drive/MyDrive/YTShortClipper/output')
if output_dir.exists():
    for date_dir in sorted(output_dir.iterdir(), reverse=True):
        if date_dir.is_dir():
            files = list(date_dir.glob('*.mp4'))
            if files:
                print(f'\n📁 {date_dir.name}/')
                for f in sorted(files, key=lambda x: x.stat().st_mtime, reverse=True):
                    size_mb = f.stat().st_size / (1024*1024)
                    print(f'   🎬 {f.name} ({size_mb:.1f} MB)')

## 📌 Cell 7: Stop Server

Jalankan cell ini untuk menghentikan Streamlit server.

In [ ]:
!pkill -f streamlit 2>/dev/null || true
try:
    from pyngrok import ngrok
    ngrok.kill()
except:
    pass
print('✅ Streamlit & Ngrok stopped.')